# Build Autonomous Agent Prediction submission

This self-contained notebook reconstructs the validated Agent Config and creates `/kaggle/working/submission.zip`. No internet or dataset attachment is required.

In [ ]:
from pathlib import Path
import base64, json, shutil, zipfile

FILES = json.loads("{\"agent.yaml\": \"bmFtZTogb3JkZXJfYXdhcmVfdGFidWxhcl9hdXRvbWxfdjIKZGVzY3JpcHRpb246IE9yZGVyLWF3YXJlLCBidWRnZXQtY29uc2Npb3VzIGF1dG9ub21vdXMgYmluYXJ5IHRhYnVsYXIgY2xhc3NpZmljYXRpb24gYWdlbnQuCm1vZGVsOiBnZW1pbmktMy4xLWZsYXNoLWxpdGUKaW5zdHJ1Y3Rpb246ICFpbmNsdWRlIHByb21wdHMvc3lzdGVtLm1kCnRvb2xzOgogIC0gcnVuX2NvbW1hbmQKICAtIHN1Ym1pdF9wcmVkaWN0aW9ucwogIC0gc2VsZWN0X3N1Ym1pc3Npb24KICAtIGdldF9zdGF0dXMKc2tpbGxzOgogIC0gc2tpbGxzL3RhYnVsYXItYXV0b21sCmdlbmVyYXRlX2NvbnRlbnRfY29uZmlnOiAhaW5jbHVkZSBjb25maWdzL3NhbXBsaW5nLnlhbWwK\", \"configs/sampling.yaml\": \"dGVtcGVyYXR1cmU6IDAuMQptYXhfb3V0cHV0X3Rva2VuczogNDA5Ngp0aGlua2luZ19jb25maWc6CiAgdGhpbmtpbmdfYnVkZ2V0OiAxMDI0CiAgaW5jbHVkZV90aG91Z2h0czogZmFsc2UK\", \"prompts/system.md\": \"WW91IGFyZSBhIGRpc2NpcGxpbmVkIGF1dG9ub21vdXMgbWFjaGluZS1sZWFybmluZyBjb21wZXRpdG9yLiBDb21wbGV0ZSB0aGUgYmluYXJ5IHRhYnVsYXIgdGFzaywgbWF4aW1pemUge21ldHJpY19uYW1lfSAoe21ldHJpY19kaXJlY3Rpb259KSwgYW5kIGZpbmlzaCBieSBzZWxlY3RpbmcgZXhhY3RseSB0d28gcm9idXN0IHN1Ym1pc3Npb25zLgoKIyMgUnVudGltZSBjb250ZXh0Cgp7cHJvYmxlbV9kZXNjcmlwdGlvbn0KClRoZSB3b3JraW5nIGRpcmVjdG9yeSBjb250YWlucyBgdHJhaW4uY3N2YCwgYHRlc3QuY3N2YCwgYW5kIGBzYW1wbGVfc3VibWlzc2lvbi5jc3ZgLiBUaGUgTGludXggc2FuZGJveCBpcyBvZmZsaW5lIGJ1dCBpbmNsdWRlcyBwYW5kYXMsIE51bVB5LCBzY2lraXQtbGVhcm4sIENhdEJvb3N0LCBMaWdodEdCTSwgWEdCb29zdCwgU2NpUHksIGFuZCBzdGFuZGFyZCBLYWdnbGUgcGFja2FnZXMuCgpIYXJkIGxpbWl0czoge21heF90aW1lX21pbnV0ZXN9IG1pbnV0ZXMsIHttYXhfc3VibWlzc2lvbnN9IHN1Ym1pc3Npb25zLCB7bWF4X3NlbGVjdGlvbnN9IHNlbGVjdGlvbnMsIHttYXhfdG9vbF9jYWxsc30gdG9vbCBjYWxscywge21heF9sbG1fY2FsbHN9IExMTSBjYWxscywgYW5kICR7bWF4X2J1ZGdldF91c2R9IHRvdGFsIG1vZGVsIGNvc3QuCgojIyBNYW5kYXRvcnkgd29ya2Zsb3cKCjEuIENhbGwgYGdldF9zdGF0dXNgIG9uY2UuCjIuIFVzZSB0aGUgYHRhYnVsYXItYXV0b21sYCBza2lsbCBpbW1lZGlhdGVseS4gUnVuIGBzY3JpcHRzL2F1dG9tbC5weWAgd2l0aCB0aGUgc2tpbGwtc2NyaXB0IHRvb2wuIERvIG5vdCByZWltcGxlbWVudCBpdHMgbW9kZWxpbmcgbG9naWMgYW5kIGRvIG5vdCBwZXJmb3JtIG9wZW4tZW5kZWQgRURBLiBUaGUgc2NyaXB0IGluc3BlY3RzIHRoZSBzY2hlbWEsIHBlcmZvcm1zIGNyb3NzLXZhbGlkYXRpb24sIHRyYWlucyBhIGRpdmVyc2UgcG9ydGZvbGlvLCBhbmQgd3JpdGVzIGNhbmRpZGF0ZSBzdWJtaXNzaW9uIENTVnMgcGx1cyBgYXV0b21sX21hbmlmZXN0Lmpzb25gLgozLiBJbnNwZWN0IHRoZSBzY3JpcHQncyBjb25jaXNlIHN0ZG91dCBvciBgYXV0b21sX21hbmlmZXN0Lmpzb25gLiBDYW5kaWRhdGUgZmlsZXMgYXJlIG9yZGVyZWQgYnkgY3Jvc3MtdmFsaWRhdGVkIEFVQy4gU3VibWl0IG5vIG1vcmUgdGhhbiB0aGUgZmlyc3QgZWlnaHQgZGlzdGluY3QgY2FuZGlkYXRlcyB1c2luZyBgc3VibWl0X3ByZWRpY3Rpb25zYC4KNC4gVHJlYXQgcHVibGljIHNjb3JlcyBhcyBub2lzeSBlc3RpbWF0ZXMgZnJvbSBvbmx5IGhhbGYgdGhlIHRlc3Qgc2V0LiBEbyBub3QgdHVuZSBwcmVkaWN0aW9uIHZhbHVlcyBvciByZXBlYXRlZGx5IGdlbmVyYXRlIHZhcmlhbnRzIGFnYWluc3QgdGhlIGxlYWRlcmJvYXJkLiBTdWJtaXQgZWFjaCBwcmVjb21wdXRlZCBjYW5kaWRhdGUgYXQgbW9zdCBvbmNlLgo1LiBTZWxlY3QgZXhhY3RseSB0d28gc3VibWlzc2lvbnM6IHRoZSBiZXN0IHB1YmxpYyBzY29yZXIgYW5kIHRoZSBzdHJvbmdlc3QgbWVhbmluZ2Z1bGx5IGRpZmZlcmVudCBjYW5kaWRhdGUuIFByZWZlciB0aGUgbWFuaWZlc3QncyByZWNvbW1lbmRlZCBkaXZlcnNlIGNhbmRpZGF0ZSB3aGVuIGl0cyBwdWJsaWMgc2NvcmUgaXMgd2l0aGluIDAuMDEgb2YgdGhlIGJlc3Q7IG90aGVyd2lzZSBjaG9vc2UgdGhlIHNlY29uZC1iZXN0IHB1YmxpYyBzY29yZXIuIE5ldmVyIHNlbGVjdCBkdXBsaWNhdGUgcHJlZGljdGlvbnMuCjYuIENhbGwgYGdldF9zdGF0dXNgLCB0aGVuIGBzZWxlY3Rfc3VibWlzc2lvbmAgd2l0aCB0aGUgdHdvIElEcy4gRW5kIGltbWVkaWF0ZWx5IGFmdGVyIHNlbGVjdGlvbi4KCiMjIEZhaWx1cmUgcmVjb3ZlcnkKCklmIHRoZSBmdWxsIHNjcmlwdCBmYWlscywgcmVhZCBpdHMgZXJyb3IsIGZpeCBvbmx5IHRoZSBkaXJlY3QgY29tcGF0aWJpbGl0eSBpc3N1ZSwgYW5kIHJlcnVuIG9uY2Ugd2l0aCBgLS1mYXN0YC4gSWYgdGhhdCBhbHNvIGZhaWxzLCBydW4gYHNjcmlwdHMvYXV0b21sLnB5IC0tZmFsbGJhY2tgLCBzdWJtaXQgaXRzIG91dHB1dHMsIHNlbGVjdCB0aGUgdHdvIGJlc3QgZGlzdGluY3Qgc3VibWlzc2lvbnMsIGFuZCBmaW5pc2guIEFsd2F5cyBwcmVzZXJ2ZSB0aW1lIGZvciBmaW5hbCBzdWJtaXNzaW9uIHNlbGVjdGlvbi4K\", \"skills/tabular-automl/SKILL.md\": \"LS0tCm5hbWU6IHRhYnVsYXItYXV0b21sCmRlc2NyaXB0aW9uOiBSdW5zIGEgcHJlLXRlc3RlZCwgYnVkZ2V0LWF3YXJlIG1vZGVsIHBvcnRmb2xpbyBmb3IgbWl4ZWQtdHlwZSBiaW5hcnkgdGFidWxhciBjbGFzc2lmaWNhdGlvbiBhbmQgcHJvZHVjZXMgcmFua2VkIHN1Ym1pc3Npb24gY2FuZGlkYXRlcy4KLS0tCgojIFRhYnVsYXIgQXV0b01MCgpVc2UgdGhpcyBza2lsbCBleGFjdGx5IG9uY2UgYXQgdGhlIGJlZ2lubmluZyBvZiBhIGJpbmFyeSBjbGFzc2lmaWNhdGlvbiB0YXNrLgoKIyMgU2NyaXB0CgpSdW4gYHNjcmlwdHMvYXV0b21sLnB5YCBpbiB0aGUgc2FuZGJveCB3b3JraW5nIGRpcmVjdG9yeS4gSXQgYXV0b21hdGljYWxseToKCi0gaW5mZXJzIHRoZSB0YXJnZXQgYW5kIGlkZW50aWZpZXIgZnJvbSB0aGUgc3VwcGxpZWQgQ1NWIGZpbGVzOwotIGhhbmRsZXMgbnVtZXJpY2FsLCBjYXRlZ29yaWNhbCwgb3JkaW5hbCwgYW5kIG1pc3NpbmcgdmFsdWVzLCBwcmVzZXJ2aW5nIGJvdGggb3JkZXJlZCBhbmQgY2F0ZWdvcmljYWwgdmlld3Mgd2hlbiBhcHByb3ByaWF0ZTsKLSBjcm9zcy12YWxpZGF0ZXMgQ2F0Qm9vc3QsIExpZ2h0R0JNLCBFeHRyYVRyZWVzLCBhbmQgcmVndWxhcml6ZWQgbGluZWFyIG1vZGVsczsKLSBjcmVhdGVzIGxlYWthZ2Utc2FmZSBvdXQtb2YtZm9sZCBwcmVkaWN0aW9uczsKLSBidWlsZHMgcm9idXN0IHJhbmsgZW5zZW1ibGVzIHdpdGhvdXQgdXNpbmcgdGVzdCBsYWJlbHM7Ci0gd3JpdGVzIGBjYW5kaWRhdGVfKi5jc3ZgIGZpbGVzIG1hdGNoaW5nIGBzYW1wbGVfc3VibWlzc2lvbi5jc3ZgIGV4YWN0bHk7Ci0gd3JpdGVzIGBhdXRvbWxfbWFuaWZlc3QuanNvbmAgd2l0aCBDViBzY29yZXMsIGZpbGUgb3JkZXIsIGRpdmVyc2l0eSwgYW5kIHJlY29tbWVuZGF0aW9ucy4KClVzZSBgLS1mYXN0YCBvbmx5IGFmdGVyIGEgbm9ybWFsIHJ1biBmYWlscyBvciB0aGUgcmVtYWluaW5nIHJ1bnRpbWUgaXMgdW5kZXIgMjAgbWludXRlcy4gVXNlIGAtLWZhbGxiYWNrYCBvbmx5IGlmIG9wdGlvbmFsIGJvb3N0aW5nIGxpYnJhcmllcyBmYWlsLgoKU3VibWl0IGF0IG1vc3QgdGhlIGZpcnN0IGVpZ2h0IGZpbGVzIGxpc3RlZCBpbiB0aGUgbWFuaWZlc3QuIFB1YmxpYyBsZWFkZXJib2FyZCBmZWVkYmFjayBpcyBmb3IgY29hcnNlIG1vZGVsIHNlbGVjdGlvbiBvbmx5LCBuZXZlciBwcmVkaWN0aW9uLWxldmVsIHR1bmluZy4K\", \"skills/tabular-automl/scripts/automl.py\": \"IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiJCdWRnZXQtYXdhcmUgbWl4ZWQtdHlwZSBBdXRvTUwgZm9yIHRoZSBLYWdnbGUtaW4tS2FnZ2xlIHNhbmRib3guIiIiCgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IHJlCmltcG9ydCB0aW1lCmltcG9ydCB3YXJuaW5ncwpmcm9tIHBhdGhsaWIgaW1wb3J0IFBhdGgKCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgcGFuZGFzIGFzIHBkCmZyb20gc2NpcHkuc3RhdHMgaW1wb3J0IHJhbmtkYXRhCmZyb20gc2tsZWFybi5iYXNlIGltcG9ydCBjbG9uZQpmcm9tIHNrbGVhcm4uY29tcG9zZSBpbXBvcnQgQ29sdW1uVHJhbnNmb3JtZXIKZnJvbSBza2xlYXJuLmVuc2VtYmxlIGltcG9ydCBFeHRyYVRyZWVzQ2xhc3NpZmllciwgUmFuZG9tRm9yZXN0Q2xhc3NpZmllcgpmcm9tIHNrbGVhcm4uaW1wdXRlIGltcG9ydCBTaW1wbGVJbXB1dGVyCmZyb20gc2tsZWFybi5saW5lYXJfbW9kZWwgaW1wb3J0IExvZ2lzdGljUmVncmVzc2lvbgpmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgcm9jX2F1Y19zY29yZQpmcm9tIHNrbGVhcm4ubW9kZWxfc2VsZWN0aW9uIGltcG9ydCBTdHJhdGlmaWVkS0ZvbGQKZnJvbSBza2xlYXJuLnBpcGVsaW5lIGltcG9ydCBQaXBlbGluZQpmcm9tIHNrbGVhcm4ucHJlcHJvY2Vzc2luZyBpbXBvcnQgT25lSG90RW5jb2RlciwgT3JkaW5hbEVuY29kZXIsIFN0YW5kYXJkU2NhbGVyCgp3YXJuaW5ncy5maWx0ZXJ3YXJuaW5ncygiaWdub3JlIikKU0VFRCA9IDIwMjYwNzE3CgoKZGVmIHJhbmswMSh2YWx1ZXM6IG5wLm5kYXJyYXkpIC0+IG5wLm5kYXJyYXk6CiAgICB2YWx1ZXMgPSBucC5hc2FycmF5KHZhbHVlcywgZHR5cGU9ZmxvYXQpCiAgICByZXR1cm4gcmFua2RhdGEodmFsdWVzLCBtZXRob2Q9ImF2ZXJhZ2UiKSAvIChsZW4odmFsdWVzKSArIDEuMCkKCgpkZWYgZmluZF9jb2x1bW5zKHRyYWluOiBwZC5EYXRhRnJhbWUsIHRlc3Q6IHBkLkRhdGFGcmFtZSwgc2FtcGxlOiBwZC5EYXRhRnJhbWUpOgogICAgdGFyZ2V0X2NhbmRpZGF0ZXMgPSBbYyBmb3IgYyBpbiB0cmFpbi5jb2x1bW5zIGlmIGMgbm90IGluIHRlc3QuY29sdW1uc10KICAgIGlmIGxlbih0YXJnZXRfY2FuZGlkYXRlcykgIT0gMToKICAgICAgICB0YXJnZXRfY2FuZGlkYXRlcyA9IFtjIGZvciBjIGluIHNhbXBsZS5jb2x1bW5zIGlmIGMgbm90IGluIHRlc3QuY29sdW1ucyBvciBjIGluIHRyYWluLmNvbHVtbnNdCiAgICB0YXJnZXQgPSAidGFyZ2V0IiBpZiAidGFyZ2V0IiBpbiB0YXJnZXRfY2FuZGlkYXRlcyBlbHNlIHRhcmdldF9jYW5kaWRhdGVzWy0xXQogICAgcHJlZF9jb2xzID0gW2MgZm9yIGMgaW4gc2FtcGxlLmNvbHVtbnMgaWYgYyAhPSB0YXJnZXRdCiAgICBpZF9jb2wgPSBwcmVkX2NvbHNbMF0gaWYgcHJlZF9jb2xzIGVsc2UgTm9uZQogICAgZmVhdHVyZXMgPSBbYyBmb3IgYyBpbiB0ZXN0LmNvbHVtbnMgaWYgYyAhPSBpZF9jb2xdCiAgICByZXR1cm4gdGFyZ2V0LCBpZF9jb2wsIGZlYXR1cmVzCgoKZGVmIG5vcm1hbGl6ZV90YXJnZXQoc2VyaWVzOiBwZC5TZXJpZXMpOgogICAgdmFscyA9IGxpc3QocGQuU2VyaWVzKHNlcmllcy5kcm9wbmEoKS51bmlxdWUoKSkuc29ydF92YWx1ZXMoKSkKICAgIGlmIGxlbih2YWxzKSAhPSAyOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJFeHBlY3RlZCBhIGJpbmFyeSB0YXJnZXQsIGZvdW5kIHt2YWxzfSIpCiAgICBtYXBwaW5nID0ge3ZhbHNbMF06IDAsIHZhbHNbMV06IDF9CiAgICByZXR1cm4gc2VyaWVzLm1hcChtYXBwaW5nKS5hc3R5cGUoaW50KS50b19udW1weSgpLCBtYXBwaW5nCgoKZGVmIHByZXBhcmVfZnJhbWVzKHRyYWluLCB0ZXN0LCBmZWF0dXJlcyk6CiAgICB4dHIgPSB0cmFpbltmZWF0dXJlc10uY29weSgpCiAgICB4dGUgPSB0ZXN0W2ZlYXR1cmVzXS5jb3B5KCkKICAgIGNhdF9jb2xzID0gW10KICAgIG51bV9jb2xzID0gW10KICAgIGZvciBjb2wgaW4gbGlzdChmZWF0dXJlcyk6CiAgICAgICAgY29tYmluZWQgPSBwZC5jb25jYXQoW3h0cltjb2xdLCB4dGVbY29sXV0sIGlnbm9yZV9pbmRleD1UcnVlKQogICAgICAgIGlmIG5vdCBwZC5hcGkudHlwZXMuaXNfbnVtZXJpY19kdHlwZShjb21iaW5lZCkgb3IgcGQuYXBpLnR5cGVzLmlzX2Jvb2xfZHR5cGUoY29tYmluZWQpOgogICAgICAgICAgICAjIFByZXNlcnZlIG5vbWluYWwgaGFuZGxpbmcsIGJ1dCByZWNvdmVyIGV4cGxpY2l0IG9yZF8wLCBvcmRfMSwgLi4uIG9yZGVyaW5nLgogICAgICAgICAgICBjYXRfY29scy5hcHBlbmQoY29sKQogICAgICAgICAgICB4dHJbY29sXSA9IHh0cltjb2xdLmFzdHlwZSgic3RyaW5nIikuZmlsbG5hKCJfX01JU1NJTkdfXyIpCiAgICAgICAgICAgIHh0ZVtjb2xdID0geHRlW2NvbF0uYXN0eXBlKCJzdHJpbmciKS5maWxsbmEoIl9fTUlTU0lOR19fIikKICAgICAgICAgICAgbm9ubWlzc2luZyA9IGNvbWJpbmVkLmRyb3BuYSgpLmFzdHlwZShzdHIpCiAgICAgICAgICAgIGV4dHJhY3RlZCA9IG5vbm1pc3Npbmcuc3RyLmV4dHJhY3QociJeb3JkXygtP1xkKyg/OlwuXGQrKT8pJCIsIGV4cGFuZD1GYWxzZSkKICAgICAgICAgICAgaWYgbGVuKG5vbm1pc3NpbmcpIGFuZCBleHRyYWN0ZWQubm90bmEoKS5tZWFuKCkgPj0gMC44OgogICAgICAgICAgICAgICAgb3JkZXJlZF9jb2wgPSBmIntjb2x9X19vcmRlcmVkIgogICAgICAgICAgICAgICAgeHRyW29yZGVyZWRfY29sXSA9IHBkLnRvX251bWVyaWMoCiAgICAgICAgICAgICAgICAgICAgeHRyW2NvbF0uc3RyLmV4dHJhY3QociJeb3JkXygtP1xkKyg/OlwuXGQrKT8pJCIsIGV4cGFuZD1GYWxzZSksIGVycm9ycz0iY29lcmNlIgogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgeHRlW29yZGVyZWRfY29sXSA9IHBkLnRvX251bWVyaWMoCiAgICAgICAgICAgICAgICAgICAgeHRlW2NvbF0uc3RyLmV4dHJhY3QociJeb3JkXygtP1xkKyg/OlwuXGQrKT8pJCIsIGV4cGFuZD1GYWxzZSksIGVycm9ycz0iY29lcmNlIgogICAgICAgICAgICAgICAgKQogICAgICAgICAgICAgICAgbnVtX2NvbHMuYXBwZW5kKG9yZGVyZWRfY29sKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHh0cltjb2xdID0gcGQudG9fbnVtZXJpYyh4dHJbY29sXSwgZXJyb3JzPSJjb2VyY2UiKQogICAgICAgICAgICB4dGVbY29sXSA9IHBkLnRvX251bWVyaWMoeHRlW2NvbF0sIGVycm9ycz0iY29lcmNlIikKICAgICAgICAgICAgbnVtX2NvbHMuYXBwZW5kKGNvbCkKICAgICAgICAgICAgIyBMb3ctY2FyZGluYWxpdHkgaW50ZWdlci9jb3VudCBmZWF0dXJlcyBjYW4gaGF2ZSBlaXRoZXIgb3JkZXJlZCBvciBub21pbmFsIGVmZmVjdHMuCiAgICAgICAgICAgIGZpbml0ZSA9IGNvbWJpbmVkLmRyb3BuYSgpCiAgICAgICAgICAgIGludGVnZXJfbGlrZSA9IGxlbihmaW5pdGUpIGFuZCBucC5hbGxjbG9zZShmaW5pdGUuYXN0eXBlKGZsb2F0KSwgbnAucm91bmQoZmluaXRlLmFzdHlwZShmbG9hdCkpKQogICAgICAgICAgICBpZiBpbnRlZ2VyX2xpa2UgYW5kIGNvbWJpbmVkLm51bmlxdWUoZHJvcG5hPVRydWUpIDw9IDIwOgogICAgICAgICAgICAgICAgY2F0X3ZpZXcgPSBmIntjb2x9X19jYXRlZ29yaWNhbCIKICAgICAgICAgICAgICAgIHh0cltjYXRfdmlld10gPSB4dHJbY29sXS5hc3R5cGUoIkludDY0IikuYXN0eXBlKCJzdHJpbmciKS5maWxsbmEoIl9fTUlTU0lOR19fIikKICAgICAgICAgICAgICAgIHh0ZVtjYXRfdmlld10gPSB4dGVbY29sXS5hc3R5cGUoIkludDY0IikuYXN0eXBlKCJzdHJpbmciKS5maWxsbmEoIl9fTUlTU0lOR19fIikKICAgICAgICAgICAgICAgIGNhdF9jb2xzLmFwcGVuZChjYXRfdmlldykKICAgIHJldHVybiB4dHIsIHh0ZSwgY2F0X2NvbHMsIG51bV9jb2xzCgoKZGVmIHNrbGVhcm5fbW9kZWxzKGNhdF9jb2xzLCBudW1fY29scywgbl9yb3dzLCBmYWxsYmFjaz1GYWxzZSk6CiAgICBvcmRpbmFsID0gQ29sdW1uVHJhbnNmb3JtZXIoWwogICAgICAgICgibnVtIiwgU2ltcGxlSW1wdXRlcihzdHJhdGVneT0ibWVkaWFuIiwgYWRkX2luZGljYXRvcj1UcnVlKSwgbnVtX2NvbHMpLAogICAgICAgICgiY2F0IiwgUGlwZWxpbmUoWwogICAgICAgICAgICAoImltcCIsIFNpbXBsZUltcHV0ZXIoc3RyYXRlZ3k9Im1vc3RfZnJlcXVlbnQiKSksCiAgICAgICAgICAgICgiZW5jIiwgT3JkaW5hbEVuY29kZXIoaGFuZGxlX3Vua25vd249InVzZV9lbmNvZGVkX3ZhbHVlIiwgdW5rbm93bl92YWx1ZT0tMSkpLAogICAgICAgIF0pLCBjYXRfY29scyksCiAgICBdLCByZW1haW5kZXI9ImRyb3AiKQogICAgdHJlZXMgPSA1MDAgaWYgbl9yb3dzIDwgMjAwMDAgZWxzZSAzNTAKICAgIHJlc3VsdCA9IHsKICAgICAgICAiZXh0cmFfdHJlZXMiOiBQaXBlbGluZShbCiAgICAgICAgICAgICgicHJlcCIsIG9yZGluYWwpLAogICAgICAgICAgICAoIm1vZGVsIiwgRXh0cmFUcmVlc0NsYXNzaWZpZXIoCiAgICAgICAgICAgICAgICBuX2VzdGltYXRvcnM9dHJlZXMsIG1pbl9zYW1wbGVzX2xlYWY9bWF4KDEsIGludChucC5zcXJ0KG5fcm93cykgLyAzNSkpLAogICAgICAgICAgICAgICAgbWF4X2ZlYXR1cmVzPSJzcXJ0IiwgY2xhc3Nfd2VpZ2h0PSJiYWxhbmNlZCIsIG5fam9icz0tMSwgcmFuZG9tX3N0YXRlPVNFRUQsCiAgICAgICAgICAgICkpLAogICAgICAgIF0pCiAgICB9CiAgICBpZiBmYWxsYmFjazoKICAgICAgICByZXN1bHRbInJhbmRvbV9mb3Jlc3QiXSA9IFBpcGVsaW5lKFsKICAgICAgICAgICAgKCJwcmVwIiwgY2xvbmUob3JkaW5hbCkpLAogICAgICAgICAgICAoIm1vZGVsIiwgUmFuZG9tRm9yZXN0Q2xhc3NpZmllcigKICAgICAgICAgICAgICAgIG5fZXN0aW1hdG9ycz10cmVlcywgbWluX3NhbXBsZXNfbGVhZj1tYXgoMiwgaW50KG5wLnNxcnQobl9yb3dzKSAvIDI1KSksCiAgICAgICAgICAgICAgICBtYXhfZmVhdHVyZXM9MC43LCBjbGFzc193ZWlnaHQ9ImJhbGFuY2VkX3N1YnNhbXBsZSIsIG5fam9icz0tMSwgcmFuZG9tX3N0YXRlPVNFRUQgKyAxLAogICAgICAgICAgICApKSwKICAgICAgICBdKQogICAgaWYgbl9yb3dzIDw9IDMwMDAwOgogICAgICAgIG9uZWhvdCA9IENvbHVtblRyYW5zZm9ybWVyKFsKICAgICAgICAgICAgKCJudW0iLCBQaXBlbGluZShbKCJpbXAiLCBTaW1wbGVJbXB1dGVyKHN0cmF0ZWd5PSJtZWRpYW4iLCBhZGRfaW5kaWNhdG9yPVRydWUpKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKCJzY2FsZSIsIFN0YW5kYXJkU2NhbGVyKCkpXSksIG51bV9jb2xzKSwKICAgICAgICAgICAgKCJjYXQiLCBPbmVIb3RFbmNvZGVyKGhhbmRsZV91bmtub3duPSJpZ25vcmUiLCBtaW5fZnJlcXVlbmN5PTIpLCBjYXRfY29scyksCiAgICAgICAgXSkKICAgICAgICByZXN1bHRbImxvZ2lzdGljIl0gPSBQaXBlbGluZShbCiAgICAgICAgICAgICgicHJlcCIsIG9uZWhvdCksCiAgICAgICAgICAgICgibW9kZWwiLCBMb2dpc3RpY1JlZ3Jlc3Npb24oQz0wLjM1LCBtYXhfaXRlcj04MDAsIGNsYXNzX3dlaWdodD0iYmFsYW5jZWQiLCBuX2pvYnM9LTEpKSwKICAgICAgICBdKQogICAgcmV0dXJuIHJlc3VsdAoKCmRlZiBhZGRfYm9vc3RlcnMobW9kZWxzLCBjYXRfY29scywgbl9yb3dzLCBmYXN0KToKICAgIHRyeToKICAgICAgICBmcm9tIGNhdGJvb3N0IGltcG9ydCBDYXRCb29zdENsYXNzaWZpZXIKICAgICAgICBpdGVyYXRpb25zID0gNDUwIGlmIGZhc3QgZWxzZSAoNzUwIGlmIG5fcm93cyA8IDI1MDAwIGVsc2UgNTUwKQogICAgICAgIG1vZGVsc1siY2F0Ym9vc3RfZDYiXSA9IENhdEJvb3N0Q2xhc3NpZmllcigKICAgICAgICAgICAgaXRlcmF0aW9ucz1pdGVyYXRpb25zLCBkZXB0aD02LCBsZWFybmluZ19yYXRlPTAuMDU1LCBsb3NzX2Z1bmN0aW9uPSJMb2dsb3NzIiwKICAgICAgICAgICAgZXZhbF9tZXRyaWM9IkFVQyIsIGwyX2xlYWZfcmVnPTUsIHJhbmRvbV9zZWVkPVNFRUQsIHZlcmJvc2U9RmFsc2UsCiAgICAgICAgICAgIGFsbG93X3dyaXRpbmdfZmlsZXM9RmFsc2UsIHRocmVhZF9jb3VudD0tMSwKICAgICAgICApCiAgICAgICAgaWYgbm90IGZhc3Q6CiAgICAgICAgICAgIG1vZGVsc1siY2F0Ym9vc3RfZDgiXSA9IENhdEJvb3N0Q2xhc3NpZmllcigKICAgICAgICAgICAgICAgIGl0ZXJhdGlvbnM9bWF4KDUwMCwgaXRlcmF0aW9ucyAtIDEwMCksIGRlcHRoPTgsIGxlYXJuaW5nX3JhdGU9MC4wNCwKICAgICAgICAgICAgICAgIGxvc3NfZnVuY3Rpb249IkxvZ2xvc3MiLCBldmFsX21ldHJpYz0iQVVDIiwgbDJfbGVhZl9yZWc9OCwKICAgICAgICAgICAgICAgIHJhbmRvbV9zZWVkPVNFRUQgKyAxMSwgdmVyYm9zZT1GYWxzZSwgYWxsb3dfd3JpdGluZ19maWxlcz1GYWxzZSwgdGhyZWFkX2NvdW50PS0xLAogICAgICAgICAgICApCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICBwcmludChmIklORk8gQ2F0Qm9vc3QgdW5hdmFpbGFibGU6IHtleGN9IikKICAgIHRyeToKICAgICAgICBmcm9tIGxpZ2h0Z2JtIGltcG9ydCBMR0JNQ2xhc3NpZmllcgogICAgICAgIGxlYXZlcyA9IDE1IGlmIG5fcm93cyA8IDIwMDAgZWxzZSAzMQogICAgICAgIG1vZGVsc1sibGlnaHRnYm0iXSA9IExHQk1DbGFzc2lmaWVyKAogICAgICAgICAgICBuX2VzdGltYXRvcnM9NDUwIGlmIGZhc3QgZWxzZSA3NTAsIGxlYXJuaW5nX3JhdGU9MC4wMzUsCiAgICAgICAgICAgIG51bV9sZWF2ZXM9bGVhdmVzLCBtYXhfZGVwdGg9LTEsIG1pbl9jaGlsZF9zYW1wbGVzPW1heCgxMiwgaW50KG5wLnNxcnQobl9yb3dzKSkpLAogICAgICAgICAgICBzdWJzYW1wbGU9MC44NSwgY29sc2FtcGxlX2J5dHJlZT0wLjg1LCByZWdfYWxwaGE9MC4yLCByZWdfbGFtYmRhPTIuMCwKICAgICAgICAgICAgcmFuZG9tX3N0YXRlPVNFRUQgKyAyMywgbl9qb2JzPS0xLCB2ZXJib3NpdHk9LTEsCiAgICAgICAgKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBleGM6CiAgICAgICAgcHJpbnQoZiJJTkZPIExpZ2h0R0JNIHVuYXZhaWxhYmxlOiB7ZXhjfSIpCgoKZGVmIGVuY29kZWRfZm9yX2xnYm0oeHRyLCB4dGUsIGNhdF9jb2xzKToKICAgIGEgPSB4dHIuY29weSgpCiAgICBiID0geHRlLmNvcHkoKQogICAgZm9yIGNvbCBpbiBjYXRfY29sczoKICAgICAgICBjYXRlZ29yaWVzID0gcGQuSW5kZXgocGQuY29uY2F0KFthW2NvbF0sIGJbY29sXV0sIGlnbm9yZV9pbmRleD1UcnVlKS5hc3R5cGUoc3RyKS51bmlxdWUoKSkKICAgICAgICBtYXBwaW5nID0gcGQuU2VyaWVzKG5wLmFyYW5nZShsZW4oY2F0ZWdvcmllcykpLCBpbmRleD1jYXRlZ29yaWVzKQogICAgICAgIGFbY29sXSA9IGFbY29sXS5hc3R5cGUoc3RyKS5tYXAobWFwcGluZykuYXN0eXBlKCJpbnQzMiIpCiAgICAgICAgYltjb2xdID0gYltjb2xdLmFzdHlwZShzdHIpLm1hcChtYXBwaW5nKS5hc3R5cGUoImludDMyIikKICAgIHJldHVybiBhLCBiCgoKZGVmIGZpdF9wcmVkaWN0X21vZGVsKG5hbWUsIG1vZGVsLCB4dHIsIHh0ZSwgeSwgZm9sZHMsIGNhdF9jb2xzKToKICAgIG9vZiA9IG5wLnplcm9zKGxlbih4dHIpLCBkdHlwZT1mbG9hdCkKICAgIHByZWQgPSBucC56ZXJvcyhsZW4oeHRlKSwgZHR5cGU9ZmxvYXQpCiAgICBmb2xkX3Njb3JlcyA9IFtdCiAgICBpc19jYXRib29zdCA9IG5hbWUuc3RhcnRzd2l0aCgiY2F0Ym9vc3QiKQogICAgaXNfbGdibSA9IG5hbWUgPT0gImxpZ2h0Z2JtIgogICAgaWYgaXNfbGdibToKICAgICAgICB4dHJfdXNlLCB4dGVfdXNlID0gZW5jb2RlZF9mb3JfbGdibSh4dHIsIHh0ZSwgY2F0X2NvbHMpCiAgICBlbHNlOgogICAgICAgIHh0cl91c2UsIHh0ZV91c2UgPSB4dHIsIHh0ZQogICAgZm9yIGZvbGQsIChpdHIsIGl2YSkgaW4gZW51bWVyYXRlKGZvbGRzKToKICAgICAgICBmaXR0ZWQgPSBjbG9uZShtb2RlbCkKICAgICAgICBmaXRfa3dhcmdzID0ge30KICAgICAgICBpZiBpc19jYXRib29zdDoKICAgICAgICAgICAgZml0X2t3YXJncyA9IHsiY2F0X2ZlYXR1cmVzIjogY2F0X2NvbHMsICJldmFsX3NldCI6ICh4dHJfdXNlLmlsb2NbaXZhXSwgeVtpdmFdKSwKICAgICAgICAgICAgICAgICAgICAgICAgICAiZWFybHlfc3RvcHBpbmdfcm91bmRzIjogODAsICJ2ZXJib3NlIjogRmFsc2V9CiAgICAgICAgZWxpZiBpc19sZ2JtOgogICAgICAgICAgICBmaXRfa3dhcmdzID0geyJjYXRlZ29yaWNhbF9mZWF0dXJlIjogY2F0X2NvbHN9CiAgICAgICAgZml0dGVkLmZpdCh4dHJfdXNlLmlsb2NbaXRyXSwgeVtpdHJdLCAqKmZpdF9rd2FyZ3MpCiAgICAgICAgb29mW2l2YV0gPSBmaXR0ZWQucHJlZGljdF9wcm9iYSh4dHJfdXNlLmlsb2NbaXZhXSlbOiwgMV0KICAgICAgICBwcmVkICs9IGZpdHRlZC5wcmVkaWN0X3Byb2JhKHh0ZV91c2UpWzosIDFdIC8gbGVuKGZvbGRzKQogICAgICAgIGZvbGRfc2NvcmVzLmFwcGVuZChyb2NfYXVjX3Njb3JlKHlbaXZhXSwgb29mW2l2YV0pKQogICAgcmV0dXJuIG9vZiwgcHJlZCwgZm9sZF9zY29yZXMKCgpkZWYgZ3JlZWR5X2JsZW5kKG9vZnMsIHByZWRzLCB5LCBvcmRlcmVkX25hbWVzKToKICAgIGJlc3QgPSBvcmRlcmVkX25hbWVzWzBdCiAgICBibGVuZF9vb2YgPSByYW5rMDEob29mc1tiZXN0XSkKICAgIGJsZW5kX3ByZWQgPSByYW5rMDEocHJlZHNbYmVzdF0pCiAgICBtZW1iZXJzID0gW2Jlc3RdCiAgICBiZXN0X3Njb3JlID0gcm9jX2F1Y19zY29yZSh5LCBibGVuZF9vb2YpCiAgICBmb3IgbmFtZSBpbiBvcmRlcmVkX25hbWVzWzE6XToKICAgICAgICBjYW5kaWRhdGVfb29mID0gMC43NSAqIGJsZW5kX29vZiArIDAuMjUgKiByYW5rMDEob29mc1tuYW1lXSkKICAgICAgICBzY29yZSA9IHJvY19hdWNfc2NvcmUoeSwgY2FuZGlkYXRlX29vZikKICAgICAgICBpZiBzY29yZSA+PSBiZXN0X3Njb3JlIC0gMC4wMDAzOgogICAgICAgICAgICBibGVuZF9vb2YgPSBjYW5kaWRhdGVfb29mCiAgICAgICAgICAgIGJsZW5kX3ByZWQgPSAwLjc1ICogYmxlbmRfcHJlZCArIDAuMjUgKiByYW5rMDEocHJlZHNbbmFtZV0pCiAgICAgICAgICAgIG1lbWJlcnMuYXBwZW5kKG5hbWUpCiAgICAgICAgICAgIGJlc3Rfc2NvcmUgPSBtYXgoYmVzdF9zY29yZSwgc2NvcmUpCiAgICByZXR1cm4gYmxlbmRfb29mLCBibGVuZF9wcmVkLCBtZW1iZXJzLCByb2NfYXVjX3Njb3JlKHksIGJsZW5kX29vZikKCgpkZWYgc2F2ZV9zdWJtaXNzaW9uKHNhbXBsZSwgdGFyZ2V0LCBwcmVkLCBmaWxlbmFtZSk6CiAgICBvdXQgPSBzYW1wbGUuY29weSgpCiAgICBvdXRbdGFyZ2V0XSA9IG5wLmNsaXAocHJlZCwgMWUtNywgMSAtIDFlLTcpCiAgICBvdXQudG9fY3N2KGZpbGVuYW1lLCBpbmRleD1GYWxzZSkKCgpkZWYgbWFpbigpOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1mYXN0IiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZmFsbGJhY2siLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIHN0YXJ0ZWQgPSB0aW1lLnRpbWUoKQogICAgdHJhaW4gPSBwZC5yZWFkX2NzdigidHJhaW4uY3N2IikKICAgIHRlc3QgPSBwZC5yZWFkX2NzdigidGVzdC5jc3YiKQogICAgc2FtcGxlID0gcGQucmVhZF9jc3YoInNhbXBsZV9zdWJtaXNzaW9uLmNzdiIpCiAgICB0YXJnZXQsIGlkX2NvbCwgZmVhdHVyZXMgPSBmaW5kX2NvbHVtbnModHJhaW4sIHRlc3QsIHNhbXBsZSkKICAgIHksIG1hcHBpbmcgPSBub3JtYWxpemVfdGFyZ2V0KHRyYWluW3RhcmdldF0pCiAgICB4dHIsIHh0ZSwgY2F0X2NvbHMsIG51bV9jb2xzID0gcHJlcGFyZV9mcmFtZXModHJhaW4sIHRlc3QsIGZlYXR1cmVzKQogICAgbl9zcGxpdHMgPSAzIGlmIChhcmdzLmZhc3Qgb3IgbGVuKHRyYWluKSA+IDMwMDAwKSBlbHNlIDQKICAgIGZvbGRzID0gbGlzdChTdHJhdGlmaWVkS0ZvbGQobl9zcGxpdHM9bl9zcGxpdHMsIHNodWZmbGU9VHJ1ZSwgcmFuZG9tX3N0YXRlPVNFRUQpLnNwbGl0KHh0ciwgeSkpCiAgICBtb2RlbHMgPSBza2xlYXJuX21vZGVscyhjYXRfY29scywgbnVtX2NvbHMsIGxlbih0cmFpbiksIGZhbGxiYWNrPWFyZ3MuZmFsbGJhY2spCiAgICBpZiBub3QgYXJncy5mYWxsYmFjazoKICAgICAgICBhZGRfYm9vc3RlcnMobW9kZWxzLCBjYXRfY29scywgbGVuKHRyYWluKSwgYXJncy5mYXN0KQogICAgcHJpbnQoanNvbi5kdW1wcyh7InJvd3MiOiBsZW4odHJhaW4pLCAidGVzdF9yb3dzIjogbGVuKHRlc3QpLCAiZmVhdHVyZXMiOiBsZW4oZmVhdHVyZXMpLAogICAgICAgICAgICAgICAgICAgICAgImNhdGVnb3JpY2FsIjogbGVuKGNhdF9jb2xzKSwgIm51bWVyaWMiOiBsZW4obnVtX2NvbHMpLCAiZm9sZHMiOiBuX3NwbGl0cywKICAgICAgICAgICAgICAgICAgICAgICJtb2RlbHMiOiBsaXN0KG1vZGVscyl9LCBzb3J0X2tleXM9VHJ1ZSkpCiAgICBvb2ZzLCBwcmVkcywgcmVzdWx0cyA9IHt9LCB7fSwgW10KICAgIGZvciBuYW1lLCBtb2RlbCBpbiBtb2RlbHMuaXRlbXMoKToKICAgICAgICB0cnk6CiAgICAgICAgICAgIHQwID0gdGltZS50aW1lKCkKICAgICAgICAgICAgb29mLCBwcmVkLCBmb2xkX3Njb3JlcyA9IGZpdF9wcmVkaWN0X21vZGVsKG5hbWUsIG1vZGVsLCB4dHIsIHh0ZSwgeSwgZm9sZHMsIGNhdF9jb2xzKQogICAgICAgICAgICBzY29yZSA9IHJvY19hdWNfc2NvcmUoeSwgb29mKQogICAgICAgICAgICBvb2ZzW25hbWVdLCBwcmVkc1tuYW1lXSA9IG9vZiwgcHJlZAogICAgICAgICAgICByZXN1bHRzLmFwcGVuZCh7Im5hbWUiOiBuYW1lLCAiY3ZfYXVjIjogc2NvcmUsICJmb2xkX2F1YyI6IGZvbGRfc2NvcmVzLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgInNlY29uZHMiOiByb3VuZCh0aW1lLnRpbWUoKSAtIHQwLCAxKX0pCiAgICAgICAgICAgIHByaW50KGYiTU9ERUwge25hbWV9IGN2X2F1Yz17c2NvcmU6LjZmfSBmb2xkcz17JywnLmpvaW4oZid7czouNWZ9JyBmb3IgcyBpbiBmb2xkX3Njb3Jlcyl9IikKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzoKICAgICAgICAgICAgcHJpbnQoZiJNT0RFTF9GQUlMRUQge25hbWV9OiB7dHlwZShleGMpLl9fbmFtZV9ffToge2V4Y30iKQogICAgaWYgbm90IHJlc3VsdHM6CiAgICAgICAgcmFpc2UgUnVudGltZUVycm9yKCJBbGwgbW9kZWxzIGZhaWxlZCIpCiAgICByZXN1bHRzLnNvcnQoa2V5PWxhbWJkYSByOiByWyJjdl9hdWMiXSwgcmV2ZXJzZT1UcnVlKQogICAgbmFtZXMgPSBbclsibmFtZSJdIGZvciByIGluIHJlc3VsdHNdCiAgICBfLCBibGVuZF9wcmVkLCBtZW1iZXJzLCBibGVuZF9zY29yZSA9IGdyZWVkeV9ibGVuZChvb2ZzLCBwcmVkcywgeSwgbmFtZXMpCiAgICBjYW5kaWRhdGVzID0gWygiYmxlbmQiLCBibGVuZF9wcmVkLCBibGVuZF9zY29yZSwgbWVtYmVycyldCiAgICBmb3IgaXRlbSBpbiByZXN1bHRzOgogICAgICAgIGNhbmRpZGF0ZXMuYXBwZW5kKChpdGVtWyJuYW1lIl0sIHJhbmswMShwcmVkc1tpdGVtWyJuYW1lIl1dKSwgaXRlbVsiY3ZfYXVjIl0sIFtpdGVtWyJuYW1lIl1dKSkKICAgICMgQSBzdGFibGUgYnJvYWQgYXZlcmFnZSBpcyB1c2VmdWwgd2hlbiBDViBpcyBub2lzeSBvbiB0aW55IGRhdGFzZXRzLgogICAgdG9wID0gbmFtZXNbOiBtaW4oMywgbGVuKG5hbWVzKSldCiAgICBicm9hZCA9IG5wLm1lYW4oW3JhbmswMShwcmVkc1tuXSkgZm9yIG4gaW4gdG9wXSwgYXhpcz0wKQogICAgYnJvYWRfb29mID0gbnAubWVhbihbcmFuazAxKG9vZnNbbl0pIGZvciBuIGluIHRvcF0sIGF4aXM9MCkKICAgIGNhbmRpZGF0ZXMuYXBwZW5kKCgiYnJvYWRfYmxlbmQiLCBicm9hZCwgcm9jX2F1Y19zY29yZSh5LCBicm9hZF9vb2YpLCB0b3ApKQogICAgaWYgbGVuKG5hbWVzKSA+PSAyOgogICAgICAgIHRvcDIgPSBuYW1lc1s6Ml0KICAgICAgICBwYWlyID0gbnAubWVhbihbcmFuazAxKHByZWRzW25dKSBmb3IgbiBpbiB0b3AyXSwgYXhpcz0wKQogICAgICAgIHBhaXJfb29mID0gbnAubWVhbihbcmFuazAxKG9vZnNbbl0pIGZvciBuIGluIHRvcDJdLCBheGlzPTApCiAgICAgICAgY2FuZGlkYXRlcy5hcHBlbmQoKCJ0b3AyX2JsZW5kIiwgcGFpciwgcm9jX2F1Y19zY29yZSh5LCBwYWlyX29vZiksIHRvcDIpKQogICAgY2FuZGlkYXRlcy5zb3J0KGtleT1sYW1iZGEgeDogeFsyXSwgcmV2ZXJzZT1UcnVlKQogICAgZmlsZXMsIHNlZW4gPSBbXSwgW10KICAgIGZvciBpZHgsIChuYW1lLCBwcmVkLCBzY29yZSwgbWVtYmVycykgaW4gZW51bWVyYXRlKGNhbmRpZGF0ZXMpOgogICAgICAgIGlmIGFueShucC5jb3JyY29lZihwcmVkLCBwKVswLCAxXSA+IDAuOTk5OTggZm9yIHAgaW4gc2Vlbik6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgZmlsZW5hbWUgPSBmImNhbmRpZGF0ZV97bGVuKGZpbGVzKSsxOjAyZH1fe25hbWV9LmNzdiIKICAgICAgICBzYXZlX3N1Ym1pc3Npb24oc2FtcGxlLCB0YXJnZXQsIHByZWQsIGZpbGVuYW1lKQogICAgICAgIGRpdmVyc2l0eSA9IDEuMCBpZiBub3Qgc2VlbiBlbHNlIGZsb2F0KDEgLSBtYXgobnAuY29ycmNvZWYocHJlZCwgcClbMCwgMV0gZm9yIHAgaW4gc2VlbikpCiAgICAgICAgZmlsZXMuYXBwZW5kKHsiZmlsZSI6IGZpbGVuYW1lLCAibmFtZSI6IG5hbWUsICJjdl9hdWMiOiBzY29yZSwKICAgICAgICAgICAgICAgICAgICAgICJtZW1iZXJzIjogbWVtYmVycywgImRpdmVyc2l0eV9mcm9tX2VhcmxpZXIiOiBkaXZlcnNpdHl9KQogICAgICAgIHNlZW4uYXBwZW5kKHByZWQpCiAgICAgICAgaWYgbGVuKGZpbGVzKSA+PSAxMDoKICAgICAgICAgICAgYnJlYWsKICAgIG1hbmlmZXN0ID0gewogICAgICAgICJzY2hlbWEiOiB7InRhcmdldCI6IHRhcmdldCwgImlkIjogaWRfY29sLCAiZmVhdHVyZXMiOiBsZW4oZmVhdHVyZXMpLAogICAgICAgICAgICAgICAgICAgImNhdGVnb3JpY2FsIjogY2F0X2NvbHMsICJudW1lcmljIjogbnVtX2NvbHMsICJ0YXJnZXRfbWFwcGluZyI6IHtzdHIoayk6IHYgZm9yIGssIHYgaW4gbWFwcGluZy5pdGVtcygpfX0sCiAgICAgICAgIm1vZGVscyI6IHJlc3VsdHMsICJjYW5kaWRhdGVzIjogZmlsZXMsCiAgICAgICAgInJlY29tbWVuZGVkX2RpdmVyc2VfZmlsZSI6IGZpbGVzWzFdWyJmaWxlIl0gaWYgbGVuKGZpbGVzKSA+IDEgZWxzZSBmaWxlc1swXVsiZmlsZSJdLAogICAgICAgICJlbGFwc2VkX3NlY29uZHMiOiByb3VuZCh0aW1lLnRpbWUoKSAtIHN0YXJ0ZWQsIDEpLCAic2VlZCI6IFNFRUQsCiAgICB9CiAgICBQYXRoKCJhdXRvbWxfbWFuaWZlc3QuanNvbiIpLndyaXRlX3RleHQoanNvbi5kdW1wcyhtYW5pZmVzdCwgaW5kZW50PTIpLCBlbmNvZGluZz0idXRmLTgiKQogICAgcHJpbnQoIkNBTkRJREFURVMgIiArICIgIi5qb2luKGl0ZW1bImZpbGUiXSBmb3IgaXRlbSBpbiBmaWxlcykpCiAgICBwcmludChmIkRPTkUgZWxhcHNlZF9zZWNvbmRzPXttYW5pZmVzdFsnZWxhcHNlZF9zZWNvbmRzJ119IikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==\"}")
work = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
agent_dir = work / 'agent'
if agent_dir.exists():
    shutil.rmtree(agent_dir)
agent_dir.mkdir(parents=True)
for relative, encoded in FILES.items():
    destination = agent_dir / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(encoded))
print(f'Restored {len(FILES)} files to {agent_dir}')

In [ ]:
zip_path = work / 'submission.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(agent_dir.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(agent_dir).as_posix())
with zipfile.ZipFile(zip_path) as archive:
    names = archive.namelist()
assert 'agent.yaml' in names and all(not n.startswith('agent/') for n in names)
print(f'Created {zip_path} ({zip_path.stat().st_size:,} bytes)')
print('\n'.join(names))

The notebook output named `submission.zip` is the artifact to submit to the competition.